# EchoAlign: per-pixel segmentation from frozen patch tokens

This notebook is a **demo** of the segmentation evaluation in `eval/segment`. It loads one CAMUS
frame, extracts EchoAlign's 14 x 14 patch-token map of that frame, and fits a linear head on a
handful of CAMUS patients. The numbers it prints come from a few images and a short schedule; they
are **not** the paper's results.

To reproduce the paper's segmentation tables (CAMUS, EchoNet-Dynamic, EchoNet-Pediatric and the
CAMUS-to-EchoNet transfer, for EchoAlign, EchoPrime, PanEcho and EchoCLIP), run
`python scripts/segment.py` (see its `--help`).

In [ ]:
# Set these before running.
camus_root = '/path/to/CAMUS_public'        # folder with database_nifti/ and database_split/
weights_path = '../weights/EchoAlign.pt'
patient, view = 'patient0001', '4CH'        # the frame shown in section 1
n_train_patients, n_val_patients = 4, 2      # tiny head-training demo in section 2
device = None                                # None: GPU when available, otherwise CPU

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from eval.segment import data as D, features as FE, heads as H, metrics as M

encoder = FE.FeatureMaps('echoalign', device=device, weights=weights_path)
print('device:', encoder.device)

## 1. One frame and its 14 x 14 patch-token map

`camus_clip` returns the clip's manifest row and the expert ED and ES masks (0 background, 1 LV
cavity, 2 LV myocardium, 3 LA), in the orientation `echoalign.loader.read_frames` gives the
frames. `model_input` reads 16 frames at 24 fps (consecutive frames when the file is not a video with a stored frame rate,
as for the CAMUS NIfTI files) that contain the annotated frame
(at position 8 when the clip is long enough) and resizes the frames to 224 x 224.

EchoAlign returns 196 tokens per frame from the TimeSformer's last hidden state (16 x 16-pixel
patches, a 14 x 14 grid). `FeatureMaps.maps` keeps the annotated frame's tokens as a
768 x 14 x 14 map. The same tokens are available from the model class:
`EchoAlign.encode_clips(clips, return_tokens=True)` and `EchoAlign.patch_token_map`.

In [ ]:
row, ed_mask, es_mask = D.camus_clip(camus_root, patient, view)
item = dict(video_path=row['video_path'], n=row['n_frames'], a=row['ed_idx'], row=0)
x, pos, labels224 = FE.model_input('echoalign', item, ed_mask)
fmap = encoder.maps(x[None], [pos])[0].cpu()              # [768, 14, 14]
print('frames', row['n_frames'], 'ED frame', row['ed_idx'], 'window position', pos, 'map', tuple(fmap.shape))

# Show the map as colour: its first three principal components over the 196 patches.
t = fmap.reshape(768, -1).T.numpy()
t = t - t.mean(0)
pcs = t @ np.linalg.svd(t, full_matrices=False)[2][:3].T
pcs = (pcs - pcs.min(0)) / (pcs.max(0) - pcs.min(0) + 1e-8)
frame = D.read_at(row['video_path'], [row['ed_idx']])[0]

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(frame); ax[0].set_title(f'{patient} {view} ED frame')
ax[1].imshow(ed_mask, cmap='viridis', vmin=0, vmax=3); ax[1].set_title('expert mask')
ax[2].imshow(pcs.reshape(14, 14, 3)); ax[2].set_title('patch tokens (3 principal components)')
for a in ax:
    a.axis('off')
plt.show()

## 2. Fit and apply a linear head (tiny demo)

The linear head is BatchNorm then a 1 x 1 convolution on the 14 x 14 map, with logits bilinearly
resized to 224 x 224 (`heads.make_head('linear', 768, 4)`, 4,612 parameters). `heads.fit` trains
one head per learning rate, scores the VAL images after every epoch at their native size, and
keeps the best (learning rate, epoch). Here: a few patients from the official CAMUS split, two
learning rates and 10 epochs. The paper uses every patient, 7 learning rates and 30 epochs, and
adds a shared decoder (`make_head('decoder', ...)`) trained with augmentation.

In [ ]:
split_dir = Path(camus_root) / 'database_split'
train_p = (split_dir / 'subgroup_training.txt').read_text().split()[:n_train_patients]
val_p = (split_dir / 'subgroup_validation.txt').read_text().split()[:n_val_patients]

feats, labs, splits, masks, shapes = [], [], [], [], []
for split, plist in (('TRAIN', train_p), ('VAL', val_p)):
    for p in plist:
        for v in ('2CH', '4CH'):
            r, ed, es = D.camus_clip(camus_root, p, v, split)
            for phase, m in (('ed', ed), ('es', es)):
                it = dict(video_path=r['video_path'], n=r['n_frames'], a=r[f'{phase}_idx'], row=len(feats))
                x, pos, lab = FE.model_input('echoalign', it, m)
                feats.append(encoder.maps(x[None], [pos])[0].cpu().half())
                labs.append(torch.from_numpy(lab))
                splits.append(split)
                masks.append(torch.from_numpy(m))
                shapes.append(m.shape)
feat, lab = torch.stack(feats), torch.stack(labs)
row, ver, splits = np.arange(len(feat)), np.zeros(len(feat), int), np.array(splits)
va = np.flatnonzero(splits == 'VAL')
print('images:', {s: int((splits == s).sum()) for s in ('TRAIN', 'VAL')}, 'features', tuple(feat.shape))

In [ ]:
head, info = H.fit('linear', 'echoalign', feat, lab, row, ver, splits, [masks[i] for i in va], [shapes[i] for i in va],
                   n_cls=4, lr_grid=[1e-3, 1e-2], epochs=10, device=encoder.device, log=lambda *a: None)
print('parameters:', info['params'], '| chosen lr', info['chosen_lr'], 'epoch', info['chosen_epoch'],
      '| VAL foreground Dice (demo, not a paper result): %.3f' % info['val_fg_dice'])

In [ ]:
# Apply the head to one VAL image: logits -> native mask size -> argmax.
i = int(va[0])
pred = H.predict(head, 'linear', 'echoalign', feat, [i], [shapes[i]], encoder.device)[0].cpu().numpy()
dice, _ = M.dice_iou(pred, masks[i].numpy(), 4)
print({c: round(float(dice[k]), 3) for k, c in enumerate(D.CLASSES['camus']) if k})

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(masks[i], cmap='viridis', vmin=0, vmax=3); ax[0].set_title('expert')
ax[1].imshow(pred, cmap='viridis', vmin=0, vmax=3); ax[1].set_title('linear head on 14 x 14 tokens')
for a in ax:
    a.axis('off')
plt.show()